# 25장 공개 언어모델 다루기 — 불러오고, 얼리고, 조금만 학습하기

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch25_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `p01`, `p02` 를 치고 실행(`p02` 는 10분 이상) | [ch25_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch25_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | `p02` 의 A·B에 "얼린 것 / 학습하는 것 / 그 개수"를 주석으로 | [ch25_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch25_2_comment.ipynb) |
| 3. 다시 만들기 | 18장 `p02_finetune.py` 와 이 장 `p02` 를 나란히 놓고 같은 구조로 정리한 뒤, 14장 패키지에 `--model koelectra --lora` 로 등록 | [ch25_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch25_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "25"))                      # 이 장의 코드 폴더로 이동
if "google.colab" in sys.modules:                    # Colab: peft 설치, 충돌하는 옛 torchao 제거 (25장 「자주 막히는 곳」)
    !pip install -q peft
    !pip uninstall -y -q torchao
print("준비 끝. 작업 폴더: code/25")

준비 끝. 작업 폴더: code/25


## `p01_load_model.py` — 25.1 공개 언어 모델 불러오기: Hugging Face 의 한국어 소형 모델(KoELECTRA-small, 약 1,400만 파라미터)

처음 실행하면 모델(약 50MB)을 내려받습니다.

In [ ]:
# 25.1 공개 언어 모델 불러오기: Hugging Face 의 한국어 소형 모델(KoELECTRA-small, 약 1,400만 파라미터)
# 준비: pip install transformers    (모델은 첫 실행 때 내려받아 ~/.cache/huggingface 에 둔다)
import torch
from transformers import AutoTokenizer, AutoModel

NAME = "monologg/koelectra-small-v3-discriminator"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModel.from_pretrained(NAME)
model.eval()

print(f"모델 {NAME}")
print(f"파라미터 {sum(p.numel() for p in model.parameters()):,} | 어휘 {tok.vocab_size:,}")

…\envs\jucos\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 37577.28it/s]


[transformers] ElectraModel LOAD REPORT from: monologg/koelectra-small-v3-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


모델 monologg/koelectra-small-v3-discriminator
파라미터 14,056,192 | 어휘 35,000


In [ ]:
# ---- 토크나이저: 19장의 공백 단위와 달리 '서브워드' 로 쪼갠다. 모르는 단어가 거의 없다.
s = "제4조(성년) 사람은 19세로 성년에 이르게 된다."
enc = tok(s, return_tensors="pt")
print("\n원문  :", s)
print("토큰  :", tok.convert_ids_to_tokens(enc["input_ids"][0]))
print("번호  :", enc["input_ids"][0].tolist())
print("shape :", tuple(enc["input_ids"].shape), "(배치 1, 토큰 수)")


원문  : 제4조(성년) 사람은 19세로 성년에 이르게 된다.
토큰  : ['[CLS]', '제', '##4', '##조', '(', '성년', ')', '사람', '##은', '19', '##세', '##로', '성년', '##에', '이르', '##게', '된다', '.', '[SEP]']
번호  : [2, 3288, 4204, 4084, 12, 26556, 13, 6226, 4112, 6238, 4246, 4239, 26556, 4073, 6952, 4325, 6236, 18, 3]
shape : (1, 19) (배치 1, 토큰 수)


In [ ]:
# ---- 모델 통과: 토큰마다 벡터(은닉 상태)가 나온다. 19장 임베딩 → 22장 인코더 블록을 지난 결과.
with torch.no_grad():
    out = model(**enc)
h = out.last_hidden_state
print("\n출력 shape:", tuple(h.shape), "= [배치, 토큰 수, 차원]")


출력 shape: (1, 19, 256) = [배치, 토큰 수, 차원]


In [ ]:
# ---- 구조: 22장에서 만든 것과 같은 부품 이름이 보인다
for name, child in model.named_children():
    print(f"  {name:<11} {child.__class__.__name__}")
print("  encoder 층 수:", len(model.encoder.layer))
layer0 = model.encoder.layer[0]
for name, child in layer0.named_children():
    print(f"    layer[0].{name:<13} {child.__class__.__name__}")

  embeddings  ElectraEmbeddings
  embeddings_project Linear
  encoder     ElectraEncoder
  encoder 층 수: 12
    layer[0].attention     ElectraAttention
    layer[0].intermediate  ElectraIntermediate
    layer[0].output        ElectraOutput


In [ ]:
# ---- 문장 벡터: [CLS] 토큰(첫 번째) 또는 평균. 두 조문의 유사도
def sent_vec(text):
    e = tok(text, return_tensors="pt")
    with torch.no_grad():
        return model(**e).last_hidden_state[0].mean(0)

pairs = [("사람은 19세로 성년에 이르게 된다.", "미성년자가 법률행위를 함에는 법정대리인의 동의를 얻어야 한다."),
         ("사람은 19세로 성년에 이르게 된다.", "검사는 공소를 제기하고 수행한다.")]
for a, b in pairs:
    va, vb = sent_vec(a), sent_vec(b)
    cos = torch.cosine_similarity(va, vb, dim=0).item()
    print(f"\n유사도 {cos:.3f} : {a[:18]}… ↔ {b[:18]}…")


유사도 0.915 : 사람은 19세로 성년에 이르게 된… ↔ 미성년자가 법률행위를 함에는 법정…

유사도 0.919 : 사람은 19세로 성년에 이르게 된… ↔ 검사는 공소를 제기하고 수행한다.…


## `p02_finetune_lora.py` — 25.2 공개 언어 모델을 내 과제(법령 조문 4분류)에 맞추기: (A) 얼리고 분류층만 / (B) LoRA

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

CPU 에서는 10분 이상 걸립니다.

```python
# 25.2 공개 언어 모델을 내 과제(법령 조문 4분류)에 맞추기: (A) 얼리고 분류층만 / (B) LoRA
# 준비: pip install transformers peft
import os
import sys
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "19"))
from statutes_dataset import StatuteDataset                     # 원문·정답 목록만 빌려 쓴다 (토큰화는 HF 토크나이저로)

NAME = "monologg/koelectra-small-v3-discriminator"
LAWS = ["민법", "형법", "민사소송법", "형사소송법"]
tok = AutoTokenizer.from_pretrained(NAME)
torch.manual_seed(0)
raw_train = StatuteDataset(LAWS, "train")
raw_test = StatuteDataset(LAWS, "test", vocab=raw_train.vocab)


class HFDataset(Dataset):
    """조문 원문을 HF 토크나이저로 인코딩. 15·19장의 Dataset 뼈대 그대로."""

    def __init__(self, items, max_len=64):
        self.items, self.max_len = items, max_len

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        text, label = self.items[i]
        enc = tok(text, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
        return enc["input_ids"][0], enc["attention_mask"][0], label


train_loader = DataLoader(HFDataset(raw_train.items), batch_size=32, shuffle=True)
test_loader = DataLoader(HFDataset(raw_test.items), batch_size=128)


def accuracy(model):
    model.eval()
    c = 0
    with torch.no_grad():
        for ids, mask, y in test_loader:
            c += (model(input_ids=ids, attention_mask=mask).logits.argmax(1) == y).sum().item()
    model.train()
    return c / len(raw_test)


def run(name, model, params, lr, epochs=int(os.environ.get("EPOCHS", 2))):   # EPOCHS 환경변수로 조절
    opt = torch.optim.AdamW(params, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    n_train = sum(p.numel() for p in params)
    n_all = sum(p.numel() for p in model.parameters())
    t0 = time.time()
    for ep in range(1, epochs + 1):
        for ids, mask, y in train_loader:
            opt.zero_grad()
            loss_fn(model(input_ids=ids, attention_mask=mask).logits, y).backward()
            opt.step()
        print(f"  {name} 에포크 {ep}: 시험 정확도 {accuracy(model):.3f}  ({time.time() - t0:.0f}초)")
    print(f"{name:<18} 학습 파라미터 {n_train:>10,} / 전체 {n_all:,} ({100 * n_train / n_all:.2f}%)")


if __name__ == "__main__":
    # A. 얼리고 분류층만 (18장의 특징 추출)
    torch.manual_seed(0)
    m = AutoModelForSequenceClassification.from_pretrained(NAME, num_labels=len(LAWS))
    for p in m.parameters():
        p.requires_grad = False
    for p in m.classifier.parameters():
        p.requires_grad = True
    run("A. 분류층만", m, list(m.classifier.parameters()), lr=1e-3)

    # B. LoRA: 어텐션의 query·value 선형층 옆에 작은 저계수 행렬을 붙여 그것만 학습
    torch.manual_seed(0)
    m = AutoModelForSequenceClassification.from_pretrained(NAME, num_labels=len(LAWS))
    cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["query", "value"], lora_dropout=0.1,
                     task_type="SEQ_CLS")
    m = get_peft_model(m, cfg)
    trainable = [p for p in m.parameters() if p.requires_grad]
    run("B. LoRA(r=8)", m, trainable, lr=5e-4)
    m.print_trainable_parameters()
```

In [ ]:
!python p02_finetune_lora.py

  A. 분류층만 에포크 1: 시험 정확도 0.418  (15초)
  A. 분류층만 에포크 2: 시험 정확도 0.508  (44초)
A. 분류층만            학습 파라미터     66,820 / 전체 14,123,012 (0.47%)
  B. LoRA(r=8) 에포크 1: 시험 정확도 0.639  (52초)
  B. LoRA(r=8) 에포크 2: 시험 정확도 0.827  (116초)
B. LoRA(r=8)       학습 파라미터    165,124 / 전체 14,288,136 (1.16%)
trainable params: 165,124 || all params: 14,288,136 || trainable%: 1.1557



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 42882.58it/s]
[transformers] ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-small-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initi

## 다음 단계

- 2단계 「주석 달기」: [ch25_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch25_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch25_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch25_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.